In [3]:
## STEP 1 : Corpus 
CORPUS = {
    "d1": "Water damage to the kitchen ceiling caused by a burst pipe is covered under the standard homeowners policy.",
    "d2": "Flood damage from rising external water is excluded under the standard policy and requires separate flood coverage.",
    "d3": "The deductible for water damage claims under the standard policy is $500 per incident.",
    "d4": "Fire damage to the garage roof after a lightning strike is covered under the standard policy.",
    "d5": "The deductible for fire damage claims is $1000 per incident, higher than water damage.",
}

for doc_id, text in CORPUS.items():
    print(doc_id, "->", text)


d1 -> Water damage to the kitchen ceiling caused by a burst pipe is covered under the standard homeowners policy.
d2 -> Flood damage from rising external water is excluded under the standard policy and requires separate flood coverage.
d3 -> The deductible for water damage claims under the standard policy is $500 per incident.
d4 -> Fire damage to the garage roof after a lightning strike is covered under the standard policy.
d5 -> The deductible for fire damage claims is $1000 per incident, higher than water damage.


In [4]:
## STEP 2 : A Simple Retriever
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

doc_ids = list(CORPUS.keys())
vectorizer = TfidfVectorizer(stop_words="english")
doc_matrix = vectorizer.fit_transform(CORPUS.values())

def retrieve(query, k=3):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, doc_matrix)[0]
    ranked = sims.argsort()[::-1][:k]
    return [doc_ids[i] for i in ranked]

query = "what is the deductible for water damage"
top = retrieve(query, k=3)
print("query:", query)
print("top-3:", top)
for doc_id in top:
    print(" ", doc_id, "->", CORPUS[doc_id])


query: what is the deductible for water damage
top-3: ['d5', 'd3', 'd1']
  d5 -> The deductible for fire damage claims is $1000 per incident, higher than water damage.
  d3 -> The deductible for water damage claims under the standard policy is $500 per incident.
  d1 -> Water damage to the kitchen ceiling caused by a burst pipe is covered under the standard homeowners policy.


In [6]:
## STEP 3: Golden Query Set
GOLDEN = [
    {"query": "what is the deductible for water damage",           "relevant": {"d3"}},
    {"query": "is flood damage covered under the standard policy", "relevant": {"d2"}},
    {"query": "deductible for fire damage claims",                 "relevant": {"d5"}},
]

for item in GOLDEN:
    print(item["query"], "->", item["relevant"])


what is the deductible for water damage -> {'d3'}
is flood damage covered under the standard policy -> {'d2'}
deductible for fire damage claims -> {'d5'}


In [ ]:
## STEP 4 : The Metrics
def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / len(relevant)

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / k

def mrr(retrieved, relevant): # The rank of first relevant result
    for rank, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            return 1 / rank
    return 0.0

# manual trace, using Step 2's retriever on Step 3's first query
retrieved = retrieve("what is the deductible for water damage", k=3)
relevant = {"d3"}

print("retrieved:", retrieved)
print("Recall@3   =", recall_at_k(retrieved, relevant, 3))
print("Precision@3=", precision_at_k(retrieved, relevant, 3))
print("MRR        =", mrr(retrieved, relevant))

retrieved: ['d5', 'd3', 'd1']
Recall@3   = 1.0
Precision@3= 0.3333333333333333
MRR        = 0.5


In [ ]:
import pandas as pd

rows = []
for item in GOLDEN:
    retrieved = retrieve(item["query"], k=3)
    rows.append({
        "query": item["query"],
        "recall@k": recall_at_k(retrieved, item["relevant"], 3),
        "precision@k": precision_at_k(retrieved, item["relevant"], 3),
        "mrr": mrr(retrieved, item["relevant"]),
    })

results = pd.DataFrame(rows)
results

print("Mean across all queries:")
print(results[["recall@k", "precision@k", "mrr"]].mean())

print("\nSorted by MRR (weakest first):")
print(results.sort_values("mrr")[["query", "mrr"]].to_string(index=False))



,query,recall@k,precision@k,mrr
0,what is the deductible for water damage,1.0,0.333333,0.5
1,is flood damage covered under the standard policy,1.0,0.333333,1.0
2,deductible for fire damage claims,1.0,0.333333,1.0


In [ ]:
## ONLINE METRICS
query = "what is the deductible for water damage"
context = CORPUS["d3"]  # the doc our retriever actually surfaced as relevant

faithful_answer = "The deductible for water damage claims is $500 per incident."
hallucinated_answer = "The deductible for water damage claims is $500 per incident, and it's automatically waived for customers with more than 5 years of tenure."

print("Context:", context)
print("\nFaithful answer:", faithful_answer)
print("Hallucinated answer:", hallucinated_answer)


Context: The deductible for water damage claims under the standard policy is $500 per incident.

Faithful answer: The deductible for water damage claims is $500 per incident.
Hallucinated answer: The deductible for water damage claims is $500 per incident, and it's automatically waived for customers with more than 5 years of tenure.


In [11]:
import re

def judge_faithfulness(answer, context, overlap_threshold=0.5):
    context_words = set(re.findall(r"\w+", context.lower()))
    clauses = [c.strip() for c in re.split(r"\.|,| and ", answer) if c.strip()]

    unsupported = []
    for clause in clauses:
        clause_words = set(re.findall(r"\w+", clause.lower()))
        overlap = len(clause_words & context_words) / len(clause_words)
        if overlap < overlap_threshold:
            unsupported.append(clause)

    verdict = "FAITHFUL" if not unsupported else "UNSUPPORTED CLAIMS FOUND"
    return verdict, unsupported

for label, answer in [("faithful", faithful_answer), ("hallucinated", hallucinated_answer)]:
    verdict, unsupported = judge_faithfulness(answer, context)
    print(f"{label}: {verdict}")
    if unsupported:
        print("  unsupported clause(s):", unsupported)


faithful: FAITHFUL
hallucinated: UNSUPPORTED CLAIMS FOUND
  unsupported clause(s): ["it's automatically waived for customers with more than 5 years of tenure"]


In [12]:
ANSWERS = {
    "what is the deductible for water damage": hallucinated_answer,  # from Step 6b
    "is flood damage covered under the standard policy":
        "Flood damage is excluded under the standard policy and requires separate flood coverage.",
    "deductible for fire damage claims":
        "The deductible for fire damage claims is $1000 per incident, and claims are processed within 24 hours guaranteed.",
}

for item in GOLDEN:
    q = item["query"]
    ctx = CORPUS[list(item["relevant"])[0]]  # the labeled-relevant doc's text
    verdict, unsupported = judge_faithfulness(ANSWERS[q], ctx)
    print(f"{q}\n  -> {verdict}")
    if unsupported:
        print("     unsupported:", unsupported)


what is the deductible for water damage
  -> UNSUPPORTED CLAIMS FOUND
     unsupported: ["it's automatically waived for customers with more than 5 years of tenure"]
is flood damage covered under the standard policy
  -> FAITHFUL
deductible for fire damage claims
  -> UNSUPPORTED CLAIMS FOUND
     unsupported: ['claims are processed within 24 hours guaranteed']


In [13]:
def check_regression(current, baseline, tolerance=0.05):
    failures = []
    for metric, base_val in baseline.items():
        cur_val = current[metric]
        if cur_val < base_val - tolerance:
            failures.append(f"{metric} regressed: {cur_val:.3f} < baseline {base_val:.3f} (tol {tolerance})")
    return failures

# faithfulness rate across the golden set, from Step 6c
faithful_count = sum(
    judge_faithfulness(ANSWERS[item["query"]], CORPUS[list(item["relevant"])[0]])[0] == "FAITHFUL"
    for item in GOLDEN
)
faithful_rate = faithful_count / len(GOLDEN)

current = results[["recall@k", "precision@k", "mrr"]].mean().to_dict()
current["faithful_rate"] = faithful_rate
print("Current:", current)

baseline = {"recall@k": 1.0, "precision@k": 0.333, "mrr": 0.833, "faithful_rate": 0.333}

failures = check_regression(current, baseline)
if failures:
    print("\nREGRESSION DETECTED:")
    for f in failures:
        print(" -", f)
else:
    print("\nAll metrics within tolerance of baseline. Safe to ship.")


Current: {'recall@k': 1.0, 'precision@k': 0.3333333333333333, 'mrr': 0.8333333333333334, 'faithful_rate': 0.3333333333333333}

All metrics within tolerance of baseline. Safe to ship.
